### faiss is a library for efficient similarity search and clustering of dense vectors 

key advantage 

1. Extremly fast similarity search 
2. Memory efficient 
3. Supports GPU acceleration
4. Can handle millions of vectors


How it works 

1. Indexes vectors for last nearest neighbour search
2. Returns most similar vectors bsed on distance metrics like Euclidean or cosine similarity

In [98]:
import os 
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Langchain core imports
from langchain_core.documents import Document 
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import(
    RunnablePassthrough,
)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage,AIMessage

# Langchain specific imports

from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS 
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain 


In [99]:
sample_documents = [
    Document(
        page_content=""" Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as healthcare, finance, transportation, and entertainment. As AI technology continues to advance, it holds the potential to revolutionize many aspects of our daily lives.""",
        metadata={"source": "sample_doc_1.txt"},
    ),

    Document(
        page_content="""Machine Learning (ML) is a subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. ML algorithms can be categorized into supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, the model is trained on labeled data, while in unsupervised learning, it identifies patterns in unlabeled data. Reinforcement learning involves training an agent to make a sequence of decisions by rewarding desired behaviors. ML has numerous applications, including image and speech recognition, recommendation systems, and fraud detection.""",
        metadata={"source": "sample_doc_2.txt"}
    ),
    Document(
        page_content="""Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. Unlike supervised learning, where the model is trained on a fixed dataset, RL involves learning from the consequences of actions taken in real-time. This approach has been successfully applied in various domains, including robotics, gaming, and autonomous systems.""",
        metadata={"source": "sample_doc_3.txt"}
    ),
    Document(
        page_content="""Natural Language Processing (NLP) is a subfield of AI that focuses on the interaction between computers and humans through natural language. NLP enables machines to understand, interpret, and generate human language in a valuable way. Applications of NLP include chatbots, sentiment analysis, language translation, and information extraction.""",
        metadata={"source": "sample_doc_4.txt"}
    ),
    Document(
        page_content="""Deep Learning (DL) is a subset of machine learning that utilizes neural networks with multiple layers to model complex patterns in data. DL has been particularly successful in tasks such as image and speech recognition, natural language processing, and game playing. The ability of deep learning models to automatically learn features from raw data has led to significant advancements in AI capabilities.""",
        metadata={"source": "sample_doc_5.txt"}
    )
]
print(sample_documents)

[Document(metadata={'source': 'sample_doc_1.txt'}, page_content=' Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as healthcare, finance, transportation, and entertainment. As AI technology continues to advance, it holds the potential to revolutionize many aspects of our daily lives.'), Document(metadata={'source': 'sample_doc_2.txt'}, page_content='Machine Learning (ML) is a subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. ML algorithms can be categorized into supervised learning, unsupervised learning, and reinforcement learning. In supervised 

In [100]:
### Text splitting 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""],
    length_function=len
    )

### Split the documents into chunks 

chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(f"Number of chunks: {len(chunks)}")
print(f"First chunk content: {chunks[0].page_content}")
print(f"First chunk metadata: {chunks[0].metadata}")    

page_content='Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as healthcare, finance, transportation, and entertainment. As AI technology continues to advance,' metadata={'source': 'sample_doc_1.txt'}
Number of chunks: 7
First chunk content: Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as

In [101]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
 )

# Create FAISS vector store from documents
vectorstore = FAISS.from_documents(chunks, embeddings)

In [102]:
sample_text = "Explain the concept of reinforcement learning in machine learning."

sample_embeddings = embeddings.embed_query(sample_text)
print(f"Sample text embeddings: {sample_embeddings}")

Sample text embeddings: [0.018939299508929253, -0.08313161879777908, -0.039907388389110565, -0.02818288840353489, -0.02802809327840805, 0.05036245658993721, -0.021220028400421143, 0.01276513934135437, -0.007164934650063515, 0.05285143107175827, 0.06270850449800491, 0.061302073299884796, -0.021356629207730293, 0.019744396209716797, 0.0035477960482239723, -0.0948854461312294, 0.030628236010670662, -0.0666193813085556, 0.03751685470342636, -0.024211756885051727, -0.03545669838786125, -0.06043175235390663, -0.01972755417227745, -0.025648342445492744, -0.021513473242521286, 0.015346665866672993, 0.014331107959151268, -0.03987620398402214, -0.02061743475496769, -0.02808629721403122, 0.05322721600532532, -0.06779482960700989, 0.055422596633434296, -0.00260260421782732, 1.4652540585302631e-06, -0.0564199835062027, -0.02055293321609497, -0.006914486642926931, 0.010508544743061066, -0.0857396274805069, 0.024705849587917328, 0.03852374479174614, -0.03690492734313011, 0.021239953115582466, -0.0040

In [103]:
# Perform similarity search
similar_docs = vectorstore.similarity_search(sample_text, k=2)
 # k=2 means return top 2 similar documents

# Print the results
for doc in similar_docs:
    print("\nSimilar document content:")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)


Similar document content:
Machine Learning (ML) is a subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. ML algorithms can be categorized into supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, the model is trained on labeled data, while in unsupervised learning, it identifies patterns in unlabeled data. Reinforcement learning involves training an agent to make a sequence of decisions by

Metadata: {'source': 'sample_doc_2.txt'}

Similar document content:
Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize cumulative reward. Unlike supervised learning, where the model is trained on a fixed dataset, RL involves learning from the consequences of actions taken in real-time. This approach has been successfully applied in various domains, including robotics, gamin

In [104]:
text=["AI","Machine_Learning","Reinforcement Learning","Natural Language Processing","Deep Learning"]
batch_embeddings = embeddings.embed_documents(text)
print(f"Batch text embeddings: {batch_embeddings}")

Batch text embeddings: [[0.00802842527627945, 0.04928520321846008, -0.053655169904232025, -0.009137831628322601, -0.029928121715784073, 0.020270390436053276, 0.007680756039917469, 0.015737364068627357, 0.01486548688262701, 0.005490814335644245, 0.05760723352432251, -0.00903121754527092, -0.05884234234690666, 0.06809137016534805, 0.031829044222831726, -0.03633216768503189, 0.02042102813720703, -0.020537342876195908, 0.02746068313717842, -0.023199323564767838, -0.030417542904615402, -0.012269553728401661, -0.01549252588301897, 0.014838169328868389, -0.02552947588264942, -0.02623257227241993, -0.03043101169168949, -0.03571879118680954, 0.008162340149283409, 0.009021044708788395, -0.024465611204504967, -0.037079326808452606, -0.03349146991968155, 0.018601233139634132, 1.8759370732368552e-06, -0.02887309156358242, 0.004642374347895384, -0.0051481216214597225, -0.017725463956594467, -0.03427549824118614, -0.0058472794480621815, 0.056785088032484055, -0.007765428628772497, 0.01901079900562763

In [105]:
### Comparing embedding using cosine similarity
def compare_embeddings(text1, text2):
    emb1 = embeddings.embed_query(text1)
    emb2 = embeddings.embed_query(text2)
    cos_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return cos_sim

In [106]:
text1 = "Artificial Intelligence"
text2 = "Machine Learning"
similarity_score = compare_embeddings(text1, text2)
print(f"Cosine similarity between '{text1}' and '{text2}': {similarity_score}")

Cosine similarity between 'Artificial Intelligence' and 'Machine Learning': 0.7204509584087498


In [107]:
### create a vectore

vectorstore=FAISS.from_documents(chunks,embeddings)

vectorstore.save_local("FAISS_index")

print("vectorestore saved locally as 'FAISS_index' directory.")


### load vector store

loaded_vectorstore = FAISS.load_local("FAISS_index", embeddings,allow_dangerous_deserialization=True)
print("vectorstore loaded from 'FAISS_index' directory.")

vectorestore saved locally as 'FAISS_index' directory.
vectorstore loaded from 'FAISS_index' directory.


In [108]:
query = "What is Artificial Intelligence?"

results= vectorstore.similarity_search(query, k=2)
for doc in results:
    print("\nSimilar document content:")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)
print(results)


Similar document content:
Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as healthcare, finance, transportation, and entertainment. As AI technology continues to advance,

Metadata: {'source': 'sample_doc_1.txt'}

Similar document content:
Machine Learning (ML) is a subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. ML algorithms can be categorized into supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, the model is trained on labeled data, while in unsupervised learning, it identifies patterns in unla

In [109]:
### Similarity Search with Score 

results_with_score = vectorstore.similarity_search_with_score(query, k=2)
for doc, score in results_with_score:   
    print("\nSimilar document content:")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)
    print(f"\nSimilarity Score: {score}")


Similar document content:
Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of intelligent behavior. It encompasses a variety of subfields, including machine learning, natural language processing, robotics, and computer vision. AI systems can analyze data, recognize patterns, make decisions, and even learn from experience. The applications of AI are vast and include areas such as healthcare, finance, transportation, and entertainment. As AI technology continues to advance,

Metadata: {'source': 'sample_doc_1.txt'}

Similarity Score: 0.40900829434394836

Similar document content:
Machine Learning (ML) is a subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data. ML algorithms can be categorized into supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, the model is trained on labeled data, while in unsupervised l

In [110]:
### Search with META data filtering 

filter_dict = {"topic": "Machine learning"}
filterted_results = vectorstore.similarity_search(query, k=2, filter=filter_dict)
for doc in filterted_results:
    print("\nSimilar document content:")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)
print(filterted_results)

[]


In [111]:
#### Initilization of RAG CHAIN WITH LCEL 

import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [112]:
### LLM and RAG Chain Initialization
from langchain_groq import ChatGroq

llm = ChatGroq(
	model="llama2-70b-4096",
	temperature=0.1,
	max_retries=3
)

In [ ]:
## Simple RAG Chain with LCEL 

Simple_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the following context:"),
    ("system", "Context: {context}"),
    ("human", "{question}")
])

In [ ]:
retriver = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

In [ ]:
retriver

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024A9C58F550>, search_kwargs={'k': 2})

In [ ]:
from typing import Any, List, Mapping, Optional, Union

def format_docs(docs: List[Document]) -> str:   
    formatted = []
    for doc in docs:
        source = doc.metadata["source"]
        formatted.append(f"Source: {source}\nContent: {doc.page_content}")
    return "\n\n".join(formatted)

In [ ]:
simple_rag_chain = (
    {"context": retriver | format_docs, "question": RunnablePassthrough()} | Simple_prompt | llm | StrOutputParser()
)

In [114]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024A9C58F550>, search_kwargs={'k': 2})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Answer the question based only on the following context:'), additional_kwargs={}), SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Context: {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object 

In [115]:
### Conversational RAG Chain 

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that provides accurate information based on the provided context."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\nQuestion: {question}")
])

In [ ]:
def create_conversational_rag_chain(retriever, llm):
    """created a conversational RAG Chain with memeory"""
    return(
        RunnablePassthrough.assign(
            context= lambda x:format_docs(retriever(x["input"])),
        )
        | conversational_prompt | llm | StrOutputParser()   

    )

conversational_rag= create_conversational_rag_chain(retriver,llm)



In [ ]:
### Streaming RAG_chain 

def create_streaming_rag_chain(retriever, llm):
    """Create a streaming RAG chain"""
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()} 
        | Simple_prompt 
        | llm 
        | StrOutputParser()
    )

streaming_rag_chain = create_streaming_rag_chain(retriver, llm)

In [ ]:
llm.invoke("hi")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
def test_rag_chain(question:str):
    """Test the RAG chain with a sample query"""
    print(f"question: {question}")
    
# simple Rag chain
    print("\n--- Simple RAG Chain Response ---")
    simple_response = simple_rag_chain.invoke({"question": question})
    print(simple_response)
    